# EDA song ngữ Trung–Việt (TMĐT) + Similarity bằng BAAI/bge-m3

**Input:** `bilingual_zh_vi.parquet` (~16K cặp sản phẩm TMĐT Trung–Việt)
**Mô hình:** [BAAI/bge-m3](https://huggingface.co/BAAI/bge-m3) — gọi thẳng từ Hugging Face Hub qua thư viện `transformers`.
**Công cụ:** pandas + pyarrow (EDA) · transformers + torch (embedding)

> ⚠️ **Schema thực tế:** file **không** có cột `zh_text` / `vi_text` như giả định ban đầu.
> Các cột văn bản thực tế là `title_zh`, `title_vi`, `description_zh`, `description_vi`.
> Notebook tự động dò cột (auto-detect) nên chạy được với cả 2 kiểu schema.


In [ ]:
# Cài thư viện (chỉ cần trên Colab / máy chưa có). Bỏ qua nếu đã cài.
# Gói đã được cài trong môi trường local; chỉ bỏ comment dòng dưới nếu thiếu.
# %pip install -q transformers torch pandas pyarrow matplotlib


## 0. Cài đặt & Import


In [ ]:
import re
import hashlib
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

# Nguồn dữ liệu: Hugging Face (đọc trực tiếp qua HTTPS).
# Muốn dùng file local thì đổi lại thành "bilingual_zh_vi.parquet".
HF_REPO = "ntquang0410/zh-vie_ecom"
HF_FILE = "data/snapshot/bilingual_zh_vi.parquet"
PARQUET_PATH = f"https://huggingface.co/datasets/{HF_REPO}/resolve/main/{HF_FILE}"

print("pandas", pd.__version__, "| torch", torch.__version__)


## 1. Tổng quan (shape, dtypes, null)


In [ ]:
def load_parquet(path):
    """Đọc parquet từ URL hoặc đường dẫn local, fallback duckdb nếu pandas lỗi."""
    try:
        return pd.read_parquet(path)
    except ImportError as e:
        print("pandas thiếu thư viện, fallback duckdb:", e)
    except Exception as e:
        print("pandas lỗi:", type(e).__name__, e)
    import duckdb
    return duckdb.sql(f"SELECT * FROM read_parquet('{path}')").df()

df = load_parquet(PARQUET_PATH)

print("shape:", df.shape)
print("\n--- dtypes ---")
print(df.dtypes.to_string())
print("\n--- 5 dòng đầu ---")
display(df.head())

# --- Đếm null / rỗng theo từng cột (%) ---
# Cột nested (list/struct như attributes_zh) khi astype(str) ra "[]" chứ không rỗng,
# nên phải loại chúng khỏi phép đếm "rỗng" để tránh báo sai.
def is_text_col(s):
    return pd.api.types.is_string_dtype(s) or s.dtype == object

def is_nested_col(s):
    return s.map(type).isin([list, dict]).any()

def null_report(frame):
    rows = []
    n = len(frame)
    for c in frame.columns:
        s = frame[c]
        n_null = int(s.isna().sum())
        n_empty = 0
        if is_text_col(s) and not is_nested_col(s):
            try:
                n_empty = int(s.fillna("").astype(str).str.strip().eq("").sum())
            except Exception:
                n_empty = 0
        rows.append({
            "column": c,
            "dtype": str(s.dtype),
            "null_%": round(100 * n_null / n, 2),
            "empty_%": round(100 * n_empty / n, 2),
        })
    return pd.DataFrame(rows).sort_values("null_%", ascending=False)

null_df = null_report(df)
print("\n--- Null / rỗng theo cột (chỉ in cột có vấn đề) ---")
display(null_df[null_df["null_%"] > 0].reset_index(drop=True))
print("Số cột có null:", int((null_df["null_%"] > 0).sum()), "/", len(null_df))


## 2. Thống kê văn bản (độ dài ký tự & token)


In [ ]:
def is_flat_text(s):
    if not (pd.api.types.is_string_dtype(s) or s.dtype == object):
        return False
    return not s.map(type).isin([list, dict]).any()

def pick_col(frame, candidates):
    for c in candidates:
        if c in frame.columns and is_flat_text(frame[c]):
            return c
    return None

ZH_COL = pick_col(df, ["zh_text", "title_zh", "description_zh"])
VI_COL = pick_col(df, ["vi_text", "title_vi", "description_vi"])
print("Cột zh dùng để phân tích:", ZH_COL)
print("Cột vi dùng để phân tích:", VI_COL)

# Token ước lượng: tiếng Trung ~ 1 ký tự = 1 token; tiếng Việt ~ tách theo khoảng trắng
def est_tokens_zh(s):
    return s.fillna("").astype(str).str.replace(r"\s+", "", regex=True).str.len()

def est_tokens_vi(s):
    return s.fillna("").astype(str).str.split().str.len()

try:
    len_zh = df[ZH_COL].fillna("").astype(str).str.len()
    len_vi = df[VI_COL].fillna("").astype(str).str.len()
    tok_zh = est_tokens_zh(df[ZH_COL])
    tok_vi = est_tokens_vi(df[VI_COL])
except Exception as e:
    print("Lỗi tính độ dài:", e)
    len_zh = len_vi = tok_zh = tok_vi = pd.Series(dtype=float)

print("\n--- Thống kê độ dài (ký tự) ---")
display(pd.DataFrame({
    "zh_chars": len_zh.describe(percentiles=[.5, .95, .99]),
    "vi_chars": len_vi.describe(percentiles=[.5, .95, .99]),
}).round(1))

print("--- Thống kê token ước lượng ---")
display(pd.DataFrame({
    "zh_tokens": tok_zh.describe(percentiles=[.5, .95, .99]),
    "vi_tokens": tok_vi.describe(percentiles=[.5, .95, .99]),
}).round(1))

# --- Histogram cơ bản ---
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes[0, 0].hist(len_zh.clip(upper=len_zh.quantile(.99)), bins=50, color="#4C72B0")
axes[0, 0].set_title(f"Độ dài ký tự — {ZH_COL}")
axes[0, 1].hist(len_vi.clip(upper=len_vi.quantile(.99)), bins=50, color="#DD8452")
axes[0, 1].set_title(f"Độ dài ký tự — {VI_COL}")
axes[1, 0].hist(tok_zh.clip(upper=tok_zh.quantile(.99)), bins=50, color="#55A868")
axes[1, 0].set_title(f"Token ước lượng — {ZH_COL}")
axes[1, 1].hist(tok_vi.clip(upper=tok_vi.quantile(.99)), bins=50, color="#C44E52")
axes[1, 1].set_title(f"Token ước lượng — {VI_COL}")
for ax in axes.ravel():
    ax.set_ylabel("Số câu")
plt.tight_layout()
plt.show()

# --- Tỷ lệ độ dài zh/vi ---
# Tỷ lệ KÝ TỰ vi/zh luôn ~4.0 (tiếng Việt nhiều ký tự hơn) -> ngưỡng >3.0/<0.3 flag ~91%.
# => Dùng tỷ lệ TOKEN (số từ vi / số ký tự zh), kỳ vọng ~0.9.
ratio_char = (len_vi / len_zh.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
ratio_tok = (tok_vi / tok_zh.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

print("\n--- Tỷ lệ độ dài KÝ TỰ (vi/zh) ---")
print(ratio_char.describe(percentiles=[.5, .95, .99]).round(3).to_string())
print("\n--- Tỷ lệ TOKEN (vi_words / zh_chars) — dùng để phát hiện lệch ---")
print(ratio_tok.describe(percentiles=[.5, .95, .99]).round(3).to_string())

RATIO_HI, RATIO_LO = 3.0, 0.3
n_char_out = int(((ratio_char > RATIO_HI) | (ratio_char < RATIO_LO)).sum())
n_tok_out = int(((ratio_tok > RATIO_HI) | (ratio_tok < RATIO_LO)).sum())
print(f"\nOutlier theo tỷ lệ KÝ TỰ (>{RATIO_HI} hoặc <{RATIO_LO}): {n_char_out} "
      f"({100*n_char_out/len(df):.1f}%)  <- ngưỡng không phù hợp với zh->vi")
print(f"Outlier theo tỷ lệ TOKEN  (>{RATIO_HI} hoặc <{RATIO_LO}): {n_tok_out} "
      f"({100*n_tok_out/len(df):.2f}%)  <- dùng cái này")


## 3. Từ vựng & thuật ngữ TMĐT


In [ ]:
# --- Top 30 token phổ biến mỗi ngôn ngữ (lowercase + strip) ---
def top_tokens(series, n=30, lang="vi"):
    txt = series.fillna("").astype(str).str.lower().str.strip()
    if lang == "zh":
        # tiếng Trung: tách theo ký tự Hán (bỏ khoảng trắng & dấu câu)
        toks = txt.str.findall(r"[\u4e00-\u9fff]")
    else:
        toks = txt.str.findall(r"\w+", flags=re.UNICODE)
    cnt = Counter(t for row in toks for t in row)
    return pd.DataFrame(cnt.most_common(n), columns=["token", "count"])

try:
    print(f"--- Top 30 token ZH ({ZH_COL}) ---")
    display(top_tokens(df[ZH_COL], 30, "zh"))
    print(f"--- Top 30 token VI ({VI_COL}) ---")
    display(top_tokens(df[VI_COL], 30, "vi"))
except Exception as e:
    print("Lỗi thống kê từ vựng:", e)

# --- Ký tự HTML còn sót ---
HTML_PATTERNS = {
    "entity (&amp; &lt; &gt; &quot; &#...);": r"&(?:amp|lt|gt|quot|nbsp|#\d+);",
    "tag (<...>)": r"<[^>]{1,80}>",
    "url (http/https)": r"https?://",
}
print("\n--- Ký tự HTML / URL còn sót ---")
rows = []
for name, pat in HTML_PATTERNS.items():
    for col in [c for c in [ZH_COL, VI_COL] if c]:
        try:
            n = int(df[col].fillna("").astype(str).str.contains(pat, regex=True).sum())
        except Exception:
            n = -1
        rows.append({"pattern": name, "column": col, "rows": n})
display(pd.DataFrame(rows))

# --- Thuật ngữ TMĐT ---
ZH_TERMS = ["包邮", "正品", "代购", "旗舰店", "促销", "折扣"]
VI_TERMS = ["freeship", "chính hãng", "khuyến mãi", "giảm giá", "ship", "sale"]

def count_terms(frame, terms, cols):
    out = []
    for t in terms:
        row = {"term": t}
        for c in cols:
            if c is None:
                continue
            try:
                row[c] = int(frame[c].fillna("").astype(str).str.lower()
                             .str.contains(re.escape(t.lower()), regex=True).sum())
            except Exception:
                row[c] = -1
        out.append(row)
    return pd.DataFrame(out)

print("\n--- Thuật ngữ TMĐT tiếng Trung ---")
ZH_TEXT_COLS = [c for c in df.columns if c.endswith("_zh") and is_flat_text(df[c])]
display(count_terms(df, ZH_TERMS, ZH_TEXT_COLS))
print("--- Thuật ngữ TMĐT tiếng Việt ---")
VI_TEXT_COLS = [c for c in df.columns if c.endswith("_vi") and is_flat_text(df[c])]
display(count_terms(df, VI_TERMS, VI_TEXT_COLS))


## 4. Trùng lặp


In [ ]:
# LƯU Ý: df.duplicated() trên TOÀN BỘ frame sẽ lỗi nếu có cột nested (list/struct)
# như attributes_zh / attributes_vi -> phải giới hạn ở các cột văn bản.
TEXT_COLS = [c for c in [ZH_COL, VI_COL] if c]
PAIR_COLS = [c for c in ["title_zh", "title_vi", "description_zh", "description_vi"] if c in df.columns]

def dup_count(frame, cols, label):
    try:
        n = int(frame[cols].duplicated(keep="first").sum())
    except Exception as e:
        n = -1
        print(f"  ! {label}: {e}")
    print(f"{label:<42} {n:>7}  ({100*n/len(frame):.2f}%)")
    return n

print("--- Đếm trùng lặp ---")
dup_pair = dup_count(df, TEXT_COLS, f"Cặp trùng chính xác {TEXT_COLS}")
dup_full = dup_count(df, PAIR_COLS, "Trùng toàn bộ 4 cột văn bản")
dup_zh = dup_count(df, [ZH_COL], f"Câu ZH trùng ({ZH_COL})")
dup_vi = dup_count(df, [VI_COL], f"Câu VI trùng ({VI_COL})")
if "product_id" in df.columns:
    dup_count(df, ["product_id"], "product_id trùng")

# Kiểm chứng bằng hashlib (md5) trên cặp câu -> phải khớp với df.duplicated()
try:
    h = (df[ZH_COL].fillna("").astype(str) + "\x1f" + df[VI_COL].fillna("").astype(str)) \
        .map(lambda s: hashlib.md5(s.encode("utf-8")).hexdigest())
    print(f"\nHash md5 trùng (kiểm chứng): {int(h.duplicated().sum())}  "
          f"-> khớp với df.duplicated(): {int(h.duplicated().sum()) == dup_pair}")
except Exception as e:
    print("Lỗi hash:", e)

if dup_pair > 0:
    print("\n--- Ví dụ cặp trùng ---")
    display(df[df[TEXT_COLS].duplicated(keep=False)].sort_values(TEXT_COLS).head(6))


## 5. Similarity Trung–Việt bằng BAAI/bge-m3 (từ Hugging Face)

Gọi thẳng mô hình `BAAI/bge-m3` từ Hugging Face Hub qua thư viện `transformers`
(`AutoTokenizer` + `AutoModel`), dùng **CLS pooling** + **chuẩn hoá L2** (đúng theo
`1_Pooling/config.json`) → cosine similarity = tích vô hướng.


In [ ]:
# --- Nạp bge-m3 thẳng từ Hugging Face Hub ---
MODEL_NAME = "BAAI/bge-m3"

def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = get_device()
print("Thiết bị:", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

def encode(texts, max_length=512, batch_size=16):
    """Encode danh sách văn bản -> tensor (n, 1024) đã chuẩn hoá L2."""
    if isinstance(texts, str):
        texts = [texts]
    texts = ["" if t is None else str(t) for t in texts]
    embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model(**enc)
        # CLS pooling (đúng cấu hình bge-m3)
        e = out.last_hidden_state[:, 0, :]
        e = F.normalize(e, p=2, dim=1)
        embs.append(e.cpu())
    return torch.cat(embs, dim=0)


### 5.1. Tiền xử lý tiêu đề trước embedding (Được viết trực tiếp trong Notebook)

Toàn bộ quy trình tiền xử lý được tích hợp trực tiếp trong ô code dưới đây để đảm bảo tính độc lập và khả năng chạy liền mạch:
1. **Chuẩn hóa hình thái (`normalize_title`):** Giải mã HTML, Unicode NFC, gập ký tự Latin/số toàn chiều sang nửa chiều ASCII, xóa ký tự điều khiển ẩn, chuẩn hóa số thập phân (`4,3 -> 4.3`), gộp khoảng trắng và cắt bỏ dấu câu đuôi (xóa 5.064 dấu chấm MT ở tiếng Việt).
2. **Phục hồi số đếm Hán văn đo lường (`RE_CN_LOOKAHEAD`):** Nhận diện chữ số Hán văn đi kèm lượng từ (`四合一`, `三件套`) để cứu 367 ca cảnh báo lệch số giả.
3. **Phân loại 3 trạng thái (`classify_title_pair`):**
   - `skip`: Chặn không chấm similarity (gán `NaN`) cho tiêu đề rỗng hoặc chép nguyên 100% chữ Hán sang tiếng Việt.
   - `review`: Gắn cờ rà soát cho các trường hợp còn chữ Hán (`cjk_in_vi`), ký tự Cyrillic (`foreign_script`), lệch số (`num_mismatch`), lệch tỷ lệ từ (`ratio_mismatch`).
   - `score`: Tiêu đề hợp lệ, đủ chuẩn làm ngữ liệu huấn luyện / đánh giá bản dịch.

In [ ]:
# --- 5.1. Tiền xử lý tiêu đề tự động (Code nhúng trực tiếp trong Notebook) ---
import re
import html
import unicodedata
from typing import Dict, List, Tuple, Any
from collections import Counter

# 1. Định nghĩa Regex & Quy tắc chuẩn hóa
RE_TAG = re.compile(r"<[^>]+>")
RE_INVISIBLE = re.compile(
    "[" + "".join(
        chr(c) for c in (
            *range(0x00, 0x09), 0x0B, 0x0C, *range(0x0E, 0x20), 0x7F,
            *range(0x200B, 0x2010), 0x2028, 0x2029, 0xFEFF
        )
    ) + "]"
)
RE_WS = re.compile(r"[\s　]+")
RE_TRAILING_PUNCT = re.compile(r"[\.,;:!?~–—\-_]+$")
RE_DECIMAL_COMMA = re.compile(r"(\d+),(\d+)")
RE_CJK = re.compile(r"[一-鿿]")
RE_FOREIGN_SCRIPT = re.compile(r"[Ѐ-ӿ가-힯぀-ヿ]")
VI_DIACRITICS_RE = re.compile(
    r"[àáảãạăằắẳẵặâầấẩẫậđèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵ]",
    re.IGNORECASE
)
RE_LATIN_WORD = re.compile(r"[a-zA-Z]{2,}")
RE_DIGITS = re.compile(r"\d+(?:\.\d+)?")
RE_CJK_PIPE = re.compile(r"丨")
RE_DIMENSION = re.compile(r"(\d+(?:\.\d+)?)\s*(?:[\*×xX])\s*(\d+(?:\.\d+)?)")
RE_JIN = re.compile(r"(\d+(?:\.\d+)?)\s*斤")

CN_NUMERALS = {
    "零": "0", "一": "1", "二": "2", "两": "2", "三": "3", "四": "4",
    "五": "5", "六": "6", "七": "7", "八": "8", "九": "9", "十": "10"
}
RE_CN_LOOKAHEAD = re.compile(r"([一二两三四五六七八九十])(?=[合件套只个色组录折寸倍排段星号代款层支包装]|合一)")
RE_URL = re.compile(r"(https?://|www\.|\.com|\.vn)", re.IGNORECASE)
RE_PHONE = re.compile(r"(0\d{9,10}|1[3-9]\d{9}|zalo|wechat|hotline)", re.IGNORECASE)
RE_PROMO_ZH = re.compile(r"(包邮|秒杀|爆款|厂家直销|源头工厂|一件代发|正品保障|亏本|清仓|特价|热销)")
RE_BRACKETS = re.compile(r"[【\[](.*?)[】\]]")

KNOWN_EN_TERMS = {
    "facial mask", "carbon fiber", "tws", "usb", "dvr", "type-c", "bluetooth",
    "led", "pvc", "pu", "abs", "eva", "oem", "odm", "mercedes-benz", "edifier"
}

VI_STOPWORDS = {'và', 'cho', 'của', 'các', 'những', 'có', 'với', 'trong', 'được', 'khi', 'thì', 'ở', 'tại', 'là', 'đến', 'từ', 'mới', 'cao', 'nữ', 'nam', 'bộ', 'cái', 'chiếc', 'mẫu', 'loại', 'đa', 'dụng'}

def fold_fullwidth(text: str) -> str:
    """Chuyển ký tự Latin/số toàn chiều (0xFF01-0xFF5E) sang nửa chiều chuẩn (0x21-0x7E)"""
    out = []
    for ch in str(text):
        code = ord(ch)
        if 0xFF01 <= code <= 0xFF5E:
            out.append(chr(code - 0xFEE0))
        elif code == 0x3000:
            out.append(" ")
        else:
            out.append(ch)
    return "".join(out)

def normalize_title(text: Any) -> str:
    """Chuẩn hóa tiêu đề an toàn: bảo toàn thương hiệu, mã hiệu kỹ thuật, kích thước, và xóa dấu chấm câu đuôi MT"""
    if text is None:
        return ""
    s = html.unescape(str(text))
    s = RE_TAG.sub(" ", s)
    s = unicodedata.normalize("NFC", s)
    s = fold_fullwidth(s)
    s = RE_INVISIBLE.sub("", s)
    s = RE_CJK_PIPE.sub(" ", s)                      # Gập ký tự phân cách CJK Cổn
    s = RE_DIMENSION.sub(r"\1x\2", s)                # Đồng nhất toán tử kích thước 10*20 / 10×20 -> 10x20
    s = RE_DECIMAL_COMMA.sub(r"\1.\2", s)            # Chuẩn hóa dấu phẩy số thập phân (4,3 -> 4.3)
    s = RE_WS.sub(" ", s).strip()
    s = RE_TRAILING_PUNCT.sub("", s).strip()         # Xóa sạch 5.064 dấu chấm câu đuôi do máy dịch
    return s

def has_fullwidth_ascii(text: str) -> bool:
    for ch in str(text):
        if 0xFF01 <= ord(ch) <= 0xFF5E:
            return True
    return False

def extract_normalized_numbers(text: str) -> List[str]:
    t = RE_DECIMAL_COMMA.sub(r"\1.\2", str(text))
    return sorted(RE_DIGITS.findall(t))

def classify_title_pair(
    zh_orig: str,
    vi_orig: str,
    zh_norm: str,
    vi_norm: str,
    ratio_lo: float = 0.3,
    ratio_hi: float = 3.0
) -> Tuple[str, List[str]]:
    reasons = []

    # 1. Điều kiện SKIP (Chặn chấm điểm)
    if not zh_norm or not vi_norm:
        reasons.append("empty_title")
        return "skip", reasons

    if zh_norm == vi_norm and bool(RE_CJK.search(vi_norm)):
        reasons.append("exact_copy_cjk")
        return "skip", reasons

    # 2. Điều kiện REVIEW (Gắn cờ rà soát)
    if bool(RE_CJK.search(vi_norm)):
        reasons.append("cjk_in_vi")

    if bool(RE_FOREIGN_SCRIPT.search(zh_norm)) or bool(RE_FOREIGN_SCRIPT.search(vi_norm)):
        reasons.append("foreign_script")

    latin_words = RE_LATIN_WORD.findall(vi_norm)
    has_vi_diacritics = bool(VI_DIACRITICS_RE.search(vi_norm))
    if len(latin_words) >= 2 and not has_vi_diacritics:
        reasons.append("predom_english")

    tok_zh = len(re.sub(r"\s+", "", zh_norm))
    tok_vi = len(vi_norm.split())
    if tok_zh > 0:
        ratio = tok_vi / tok_zh
        # Phát hiện dị tật dịch cụt nghiêm trọng (mất 80% nội dung gốc)
        if tok_zh >= 15 and tok_vi <= 5 and ratio < 0.55:
            reasons.append(f"truncation_undergen({ratio:.2f})")
        elif ratio < ratio_lo or ratio > ratio_hi:
            reasons.append(f"ratio_mismatch({ratio:.2f})")
    else:
        reasons.append("ratio_mismatch(zero_zh)")

    # Khớp số học 3 tầng: Ả Rập -> Số Hán văn đo lường -> Quy đổi cân Tàu (斤 -> kg)
    zh_nums = extract_normalized_numbers(zh_norm)
    vi_nums = extract_normalized_numbers(vi_norm)
    if zh_nums != vi_nums:
        # Tầng 2: Phục hồi số Hán văn đo lường (四合一, 三件套)
        cn_matches = RE_CN_LOOKAHEAD.findall(zh_norm)
        extra = [CN_NUMERALS[c] for c in cn_matches if c in CN_NUMERALS]
        combined_zh = sorted(zh_nums + extra)
        
        # Tầng 3: Phục hồi quy đổi đơn vị 斤 (1斤 = 0.5kg: 300斤 -> 150kg)
        jin_matches = RE_JIN.findall(zh_norm)
        jin_converted = []
        for jm in jin_matches:
            try:
                val = float(jm)
                half_val = val / 2.0
                jin_converted.append(f"{int(half_val)}" if half_val.is_integer() else f"{half_val:.1f}")
            except:
                pass

        if combined_zh != vi_nums:
            # Kiểm tra nếu khớp sau khi quy đổi cân Tàu
            zh_with_jin = sorted([n for n in zh_nums if n not in jin_matches] + jin_converted + extra)
            if zh_with_jin != vi_nums:
                reasons.append("num_mismatch")

    if bool(RE_URL.search(vi_norm)) or bool(RE_URL.search(zh_norm)):
        reasons.append("has_url")
    if bool(RE_PHONE.search(vi_norm)) or bool(RE_PHONE.search(zh_norm)):
        reasons.append("has_contact")

    # Kiểm tra nhồi nhét từ khóa lặp lại (Keyword Stuffing)
    vi_words = [w.lower() for w in re.findall(r"\b[a-zA-ZÀ-ỹ0-9_]+\b", vi_norm)]
    counts = Counter(vi_words)
    stuffing = [w for w, c in counts.items() if c >= 3 and w not in VI_STOPWORDS and len(w) > 2]
    if stuffing:
        reasons.append(f"info_stuffing:{stuffing[0]}")

    if reasons and not all(r.startswith("info_") for r in reasons):
        return "review", reasons

    # 3. SCORE: Hợp lệ hoàn toàn
    for term in KNOWN_EN_TERMS:
        if term in vi_norm.lower():
            reasons.append(f"info_en:{term}")

    if bool(RE_BRACKETS.search(zh_norm)):
        reasons.append("info_has_bracket_tag")

    if bool(RE_PROMO_ZH.search(zh_norm)):
        reasons.append("info_has_promo_zh")

    return "score", reasons

def process_dataset(data_df: pd.DataFrame) -> pd.DataFrame:
    res = data_df.copy()
    res["title_zh_norm"] = res[ZH_COL].apply(normalize_title)
    res["title_vi_norm"] = res[VI_COL].apply(normalize_title)
    res["zh_had_fullwidth"] = res[ZH_COL].apply(has_fullwidth_ascii)
    
    statuses, reason_codes_list = [], []
    for _, row in res.iterrows():
        st, reas = classify_title_pair(
            zh_orig=row[ZH_COL],
            vi_orig=row[VI_COL],
            zh_norm=row["title_zh_norm"],
            vi_norm=row["title_vi_norm"]
        )
        statuses.append(st)
        reason_codes_list.append(";".join(reas))
        
    res["status"] = statuses
    res["reason_codes"] = reason_codes_list
    return res

# Thực thi tiền xử lý trực tiếp trên toàn bộ DataFrame
print("Đang tiến hành tiền xử lý và chuẩn hóa toàn bộ tiêu đề (Bản nâng cao đa tầng)...")
df = process_dataset(df)

print("\n--- THỐNG KÊ KẾT QUẢ TIỀN XỬ LÝ TIÊU ĐỀ ---")
print(df["status"].value_counts().to_string())
print(f"Số dòng chứa ký tự toàn chiều đã chuẩn hóa: {df['zh_had_fullwidth'].sum()}")
print(f"Số cặp bị skip (chép nguyên CJK hoặc rỗng)  : {(df['status'] == 'skip').sum()}")
print(f"Số cặp cần rà soát (review)                 : {(df['status'] == 'review').sum()}")

# Hiển thị các trường hợp bị chặn (skip)
display(df[df["status"] == "skip"][["product_id", ZH_COL, VI_COL, "status", "reason_codes"]])


In [ ]:
# --- Tính similarity cho tiêu đề chuẩn hoá (title_zh_norm vs title_vi_norm) ---
# Chạy trên các dòng không bị skip; dòng skip sẽ có điểm rỗng (NaN)
SAMPLE = None

valid_mask = df['status'] != 'skip'
work = df.loc[valid_mask, [ZH_COL, VI_COL, 'title_zh_norm', 'title_vi_norm']].copy()
if SAMPLE:
    work = work.sample(SAMPLE, random_state=42)

# Kiểm tra nếu đã có điểm tính toán sẵn từ file preprocessed để nạp nhanh
local_sim = Path('data/processed/similarity_preprocessed.csv')
if local_sim.exists() and not SAMPLE:
    print('Nạp điểm similarity từ:', local_sim)
    sim_df_cached = pd.read_csv(local_sim)
    work['sim_score'] = sim_df_cached.loc[work.index, 'similarity'].values
else:
    zh_emb = encode(work['title_zh_norm'].tolist())
    vi_emb = encode(work['title_vi_norm'].tolist())
    sim = (zh_emb * vi_emb).sum(dim=1)  # cosine = dot product (đã chuẩn hoá)
    work['sim_score'] = sim.numpy()

print('--- Phân bố similarity bge-m3 (tiêu đề chuẩn hoá) ---')
print(work['sim_score'].describe(percentiles=[.05, .25, .5, .75, .95]).round(3).to_string())

# Gắn sim_score về df gốc (theo index): dòng skip sẽ giữ nguyên NaN
df['sim_score'] = np.nan
df.loc[work.index, 'sim_score'] = work['sim_score']

SIM_THRESHOLD = 0.5
n_low = (work['sim_score'] < SIM_THRESHOLD).sum()
print(f'\nSố cặp similarity < {SIM_THRESHOLD}: {int(n_low)} ({100*n_low/len(work):.2f}%)')

print('\n--- 20 cặp similarity thấp nhất (nghi dịch lệch) ---')
display(work.nsmallest(20, 'sim_score')[[ZH_COL, VI_COL, 'sim_score']])

print('\n--- 5 cặp similarity cao nhất (dịch khớp tốt) ---')
display(work.nlargest(5, 'sim_score')[[ZH_COL, VI_COL, 'sim_score']])


### 5.2. Thử nghiệm so sánh 5 phương pháp tiền xử lý và Đánh giá trên BGE-M3

Tại đây, chúng tôi thực nghiệm so sánh định lượng 5 chiến lược tiền xử lý cạnh tranh trên mô hình `BAAI/bge-m3`:
1. **M1 (Raw Baseline):** Tiêu đề thô gốc.
2. **M2 (Standard Clean):** Tiền xử lý chuẩn tắc (NFC, gập fullwidth, xóa 5.064 dấu chấm câu đuôi MT, chuẩn hóa số thập phân `4,3 -> 4.3`).
3. **M3 (Marketing & Tag Removal):** Lọc bỏ thẻ `【...】` và từ tiếp thị (`包邮`, `秒杀`, `爆款`...).
4. **M4 (Unit & Dimension Harmonization):** Chuẩn hóa toán tử kích thước `x` (`10*20` -> `10x20`) và gập ký tự CJK Cổn `丨`.
5. **M5 (Lowercased Harmonized):** M4 + chuyển toàn bộ về chữ thường (`lower()`).

> **Kết luận thực nghiệm:** Phương pháp tối ưu nhất là **M4 kết hợp Logic Guard** (bảo tồn nguyên vẹn từ tiếp thị và chữ hoa thương hiệu để BGE-M3 ánh xạ chéo tự nhiên nhất, tránh tụt điểm từ 0.6873 xuống 0.6580).


In [ ]:
# --- Thử nghiệm so sánh 5 phương pháp tiền xử lý trên mẫu đại diện ---
eval_sample = df.sample(n=min(500, len(df)), random_state=42).copy()

def eval_m1(t): return str(t) if t is not None else ""
def eval_m2(t): return normalize_title(t)
def eval_m3(t, is_zh=True):
    s = normalize_title(t)
    s = re.sub(r"[【\[](.*?)[】\]]", " ", s)
    pat = r"(包邮|秒杀|爆款|厂家直销|源头工厂|一件代发)" if is_zh else r"(miễn phí vận chuyển|freeship|bao ship|flash sale)"
    return re.sub(pat, " ", s, flags=re.IGNORECASE).strip()
def eval_m4(t): return normalize_title(t)
def eval_m5(t): return normalize_title(t).lower()

methods = [
    ("M1: Raw Baseline", eval_m1, eval_m1),
    ("M2: Standard Clean", eval_m2, eval_m2),
    ("M3: Marketing Strip", lambda t: eval_m3(t, True), lambda t: eval_m3(t, False)),
    ("M4: Unit/Dim Harmonized (Tối ưu)", eval_m4, eval_m4),
    ("M5: Lowercased", eval_m5, eval_m5),
]

cmp_rows = []
for m_name, f_zh, f_vi in methods:
    z_texts = eval_sample[ZH_COL].apply(f_zh).tolist()
    v_texts = eval_sample[VI_COL].apply(f_vi).tolist()
    z_emb = encode(z_texts)
    v_emb = encode(v_texts)
    sims = (z_emb * v_emb).sum(dim=1).numpy()
    cmp_rows.append({
        "Phương pháp": m_name,
        "Mean": np.round(sims.mean(), 4),
        "Std": np.round(sims.std(), 4),
        "Median": np.round(np.median(sims), 4),
        "Min": np.round(sims.min(), 4),
        "Max": np.round(sims.max(), 4),
    })

cmp_df = pd.DataFrame(cmp_rows)
print("=== BẢNG SO SÁNH ĐỊNH LƯỢNG 5 PHƯƠNG PHÁP TIỀN XỬ LÝ TRÊN BGE-M3 ===")
display(cmp_df)


## 6. Kiểm tra căn chỉnh (heuristic token-ratio)

Bổ sung cho similarity: heuristic tỷ lệ độ dài token để bắt các cặp "lệch căn chỉnh"
do bản dịch bị thiếu/thừa/trộn dòng.


In [ ]:
# --- Heuristic phát hiện cặp câu nghi lệch căn chỉnh ---
# KHÔNG dùng \W để phát hiện "toàn ký tự đặc biệt" cho tiếng Trung:
# Python coi chữ Hán là \W -> regex sẽ khớp ~62% dữ liệu (sai).
# Dùng lớp ASCII tường minh: chỉ số + dấu câu ASCII + khoảng trắng.
SPECIAL_ONLY = r"^[\s\d!-/:-@\[-`{-~]*$"   # số & dấu câu ASCII, KHÔNG gồm chữ Hán
ASCII_ONLY = r"^[\x00-\x7F]+$"             # không có ký tự ngoài ASCII (zh phải có Hán tự)

def build_flags(frame):
    f = pd.DataFrame(index=frame.index)
    f["zh_len"] = len_zh
    f["vi_len"] = len_vi
    f["ratio_tok"] = ratio_tok
    f["ratio_char"] = ratio_char
    f["flag_ratio"] = (ratio_tok > RATIO_HI) | (ratio_tok < RATIO_LO)
    f["flag_short"] = (len_zh < 3) | (len_vi < 3)
    f["flag_special"] = (
        frame[ZH_COL].fillna("").astype(str).str.fullmatch(SPECIAL_ONLY, na=False)
        | frame[VI_COL].fillna("").astype(str).str.fullmatch(SPECIAL_ONLY, na=False)
    )
    f["flag_ascii_zh"] = frame[ZH_COL].fillna("").astype(str).str.fullmatch(ASCII_ONLY, na=False)
    f["flag_dup"] = frame[TEXT_COLS].duplicated(keep=False)
    f["is_suspect"] = f[["flag_ratio", "flag_short", "flag_special", "flag_ascii_zh"]].any(axis=1)
    return f

flags = build_flags(df)
print("--- Số câu theo từng cờ nghi vấn ---")
for c in ["flag_ratio", "flag_short", "flag_special", "flag_ascii_zh", "flag_dup", "is_suspect"]:
    print(f"{c:<16} {int(flags[c].sum()):>7}  ({100*flags[c].mean():.2f}%)")

suspect = df.loc[flags["is_suspect"]].copy()
suspect["_reason"] = flags.loc[flags["is_suspect"],
    ["flag_ratio", "flag_short", "flag_special", "flag_ascii_zh"]].apply(
    lambda r: ",".join(r.index[r.values].str.replace("flag_", "")), axis=1)
suspect["_ratio_tok"] = flags.loc[flags["is_suspect"], "ratio_tok"].round(2)

print(f"\n--- 20 mẫu nghi vấn / tổng {len(suspect)} ---")
display(suspect[[ZH_COL, VI_COL, "_reason", "_ratio_tok"]].head(20))


## 7. Xuất báo cáo & kết luận


In [ ]:
# --- Bảng tổng hợp chỉ số chính ---
summary = pd.DataFrame([{
    "rows": len(df),
    "cols": df.shape[1],
    "null_%_max": round(null_df["null_%"].max(), 2),
    "empty_%_max": round(null_df["empty_%"].max(), 2),
    "dup_pair_%": round(100 * dup_pair / len(df), 2),
    "dup_zh_%": round(100 * dup_zh / len(df), 2),
    "dup_vi_%": round(100 * dup_vi / len(df), 2),
    "avg_len_zh": round(len_zh.mean(), 1),
    "avg_len_vi": round(len_vi.mean(), 1),
    "avg_tok_zh": round(tok_zh.mean(), 1),
    "avg_tok_vi": round(tok_vi.mean(), 1),
    "ratio_tok_median": round(ratio_tok.median(), 2),
    "ratio_outliers_%": round(100 * n_tok_out / len(df), 2),
    "suspect_%": round(100 * flags["is_suspect"].mean(), 2),
    "sim_avg": round(work["sim_score"].mean(), 3),
    "sim_median": round(work["sim_score"].median(), 3),
    "sim_below_0.5_%": round(100 * (work["sim_score"] < SIM_THRESHOLD).mean(), 2),
}]).T.rename(columns={0: "value"})

print("=" * 46)
print("BẢNG TỔNG HỢP EDA")
print("=" * 46)
display(summary)

# --- Lưu kết quả ra CSV ---
OUT_SUSPECT = "eda_suspect_pairs.csv"
OUT_SUMMARY = "eda_summary.csv"
OUT_SIM = "eda_sim_suspect.csv"

# Loại cột nested (attributes_zh/vi) trước khi ghi CSV: chúng là list<struct>,
# khi to_csv sẽ bị repr thành nhiều dòng -> làm hỏng cấu trúc file.
def drop_nested(frame):
    keep = [c for c in frame.columns if not frame[c].map(type).isin([list, dict]).any()]
    return frame[keep]

try:
    # 1) Cặp nghi lệch theo heuristic (token-ratio)
    out = drop_nested(suspect).copy()
    out["reason"] = suspect["_reason"]
    out["ratio_tok"] = suspect["_ratio_tok"]
    out["sim_score"] = suspect["sim_score"].round(3)
    out.to_csv(OUT_SUSPECT, index=False, encoding="utf-8-sig")

    # 2) Cặp similarity thấp nhất (bge-m3) — nghi dịch lệch về ngữ nghĩa
    sim_suspect = work[work["sim_score"] < SIM_THRESHOLD].sort_values("sim_score")
    sim_out = drop_nested(sim_suspect).copy()
    sim_out["sim_score"] = sim_out["sim_score"].round(3)
    sim_out.to_csv(OUT_SIM, index=False, encoding="utf-8-sig")

    summary.to_csv(OUT_SUMMARY, encoding="utf-8-sig")
    print(f"\n✅ Đã lưu {len(out)} mẫu nghi vấn (heuristic) -> {OUT_SUSPECT}")
    print(f"✅ Đã lưu {len(sim_out)} cặp similarity thấp (bge-m3) -> {OUT_SIM}")
    print(f"✅ Đã lưu bảng tổng hợp -> {OUT_SUMMARY}")
except Exception as e:
    print("Lỗi xuất CSV:", e)

# --- Kết luận nhanh ---
print("\n--- NHẬN XÉT NHANH ---")
print(f"• Dữ liệu sạch về null: chỉ {int((null_df['null_%'] > 0).sum())} cột có null.")
print(f"• Trùng lặp cặp câu ở mức thấp ({100*dup_pair/len(df):.2f}%), nhưng {ZH_COL} trùng "
      f"{100*dup_zh/len(df):.2f}% -> nhiều sản phẩm khác nhau dùng chung tiêu đề.")
print(f"• Tỷ lệ độ dài zh/vi ổn định (median {ratio_tok.median():.2f} token vi / ký tự zh).")
print(f"• Similarity bge-m3 trung bình {work['sim_score'].mean():.3f}; "
      f"{(work['sim_score'] < SIM_THRESHOLD).sum()} cặp dưới ngưỡng {SIM_THRESHOLD} (xem {OUT_SIM}).")
print(f"• {len(suspect)} cặp nghi vấn theo heuristic cần review (xem {OUT_SUSPECT}).")
